In [44]:
%load_ext autoreload
%autoreload 2
from hyperparams import *
from tasks import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
# download_data()

In [46]:
mitdb, pwave = get_records(mitdb_dir, pwave_dir)

In [47]:
# 전체 데이터를 저장할 리스트 초기화
all_segments = []
all_features = []
all_labels = []
# record별 전처리
for record in mitdb:
    # load ECG signal & annotations
    signals, fields = load_ECG_signal(record)
    signals = np.squeeze(signals)
    annotations = load_ECG_annotations(record=record, dir=mitdb_dir, extension='atr')

    
    # bandpass & R-peak detection with wavelet
    bpsig = bandpass_filter(signals)
    rpeaks = get_rpeaks(bpsig) 

    # segmentation
    segments = segment_heartbeats(bpsig, rpeaks) 

    # fill Nan, IQR clipping, remove constant features
    segments = fill_nan(segments)
    segments = IQR_clipping(segments) 
    segments = remove_const_features(segments)

    # P-peak detection
    ppeaks = get_ppeaks(record, bpsig, rpeaks)
    
    # remove 1st and last R-peak & match P,R peaks
    rpeaks = rpeaks[1:-1]
    ppeaks = match_pr(rpeaks, ppeaks)

    # feature extraction for 2nd input
    extracted_features = extract_features(bpsig, ppeaks, rpeaks)


    # normalization
    extracted_features = feature_scaling(extracted_features)
    segments = feature_scaling(segments)

    # label extraction & grouping
    labels = extract_labels(rpeaks, annotations)
    labels = list(map(group_labels, labels))

    # 데이터를 리스트에 추가
    all_segments.append(segments)
    all_features.append(extracted_features)
    all_labels.append(labels)

    
x1 = np.concatenate(all_segments, axis=0)
x2 = np.concatenate(all_features, axis=0)
y = np.concatenate(all_labels, axis=0)

print("Segments(x1) Shape:", x1.shape)
print("Extracted Features(x2) Shape:", x2.shape)
print("Labels(y) Shape:", y.shape)


P-peak annotations not found for record 114. P-peaks detected using custom method.
P-peak annotations not found for record 203. P-peaks detected using custom method.
P-peak annotations not found for record 124. P-peaks detected using custom method.
P-peak annotations not found for record 230. P-peaks detected using custom method.
P-peak annotations not found for record 215. P-peaks detected using custom method.
P-peak annotations not found for record 213. P-peaks detected using custom method.
P-peak annotations not found for record 210. P-peaks detected using custom method.
P-peak annotations not found for record 109. P-peaks detected using custom method.
P-peak annotations not found for record 200. P-peaks detected using custom method.
P-peak annotations not found for record 208. P-peaks detected using custom method.
P-peak annotations not found for record 113. P-peaks detected using custom method.
P-peak annotations not found for record 209. P-peaks detected using custom method.
P-pe

In [48]:
# data split
x1_train, x1_test_val, x2_train, x2_test_val, y_train, y_test_val = train_test_split(x1, x2, y, test_size=0.3, random_state=seed, stratify=y)
x1_val, x1_test, x2_val, x2_test, y_val, y_test = train_test_split(x1_test_val, x2_test_val, y_test_val, test_size=0.5, random_state=seed, stratify=y_test_val)


In [49]:
# 각 데이터셋의 y값 분포 확인
print("Train label distribution")
print(print_label_distribution(y_train))
print("Validation label distribution")
print(print_label_distribution(y_val))
print("Test label distribution")
print(print_label_distribution(y_test))

Train label distribution
Label: Atrial, Percentage: 3.20%
Label: Normal, Percentage: 63.92%
Label: Other, Percentage: 26.82%
Label: Ventricular, Percentage: 6.07%
None
Validation label distribution
Label: Atrial, Percentage: 3.19%
Label: Normal, Percentage: 63.92%
Label: Other, Percentage: 26.82%
Label: Ventricular, Percentage: 6.06%
None
Test label distribution
Label: Atrial, Percentage: 3.20%
Label: Normal, Percentage: 63.92%
Label: Other, Percentage: 26.82%
Label: Ventricular, Percentage: 6.06%
None
